# Test runtime dependencies

This notebook tests that runtime dependencies are available in the Jupyter Notebook environment.
- Verify pandas, SciPy, ffmpeg, and ImageMagick from a user-facing Python kernel

## Parameters

In [ ]:
# Default parameters (will be overridden by Papermill)
notebook7_url = "http://localhost:8888/tree"
jupyter_token = "test-token"
default_result_path = None
close_on_fail = False
transition_timeout = 30000
jupyter_work_dir = '../artifacts/jupyter-work'

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
print(f"Created work directory: {work_dir}")

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)
import scripts.notebook7
importlib.reload(scripts.notebook7)

from scripts.playwright import *
from scripts.notebook7 import *

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## Open Jupyter Notebook and wait for it to load

In [ ]:
async def _step_wait_for_loading(page):
    await page.goto(f"{notebook7_url}?token={jupyter_token}")
    await expect(page.locator('.jp-DirListing')).to_be_visible(timeout=transition_timeout)

await run_pw(_step_wait_for_loading)

## Remove existing test notebook if it exists

In [ ]:
test_filename = "TestRuntimeDependencies.ipynb"

async def _step_remove_existing_notebook(page):
    deleted = await delete_file(page, test_filename, timeout=transition_timeout)
    if deleted:
        print("✓ Removed existing test notebook")
    else:
        print("No existing test notebook to remove")

await run_pw(_step_remove_existing_notebook)

## Create a new notebook

In [ ]:
async def _step_create_notebook(page):
    new_page = await create_new_notebook(page, kernel="Python 3", timeout=transition_timeout)
    print("✓ New notebook created")
    return new_page

await run_pw(_step_create_notebook)

## Rename and save the notebook

In [ ]:
async def _step_rename_and_save(page):
    await rename_notebook(page, test_filename, timeout=transition_timeout)
    await save_notebook(page)
    print("✓ Notebook renamed and saved")

await run_pw(_step_rename_and_save)

## Create runtime dependency checks

In [ ]:
async def _step_create_test_cells(page):
    await set_cell(
        page,
        0,
        "markdown",
        "# Runtime dependency check\n\n"
        "This notebook verifies that the runtime dependencies provided by this image "
        "are usable from the Python 3 kernel.",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "markdown",
        "## pandas\n\nVerify that pandas can create and aggregate a DataFrame.",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "code",
        "import pandas as pd\n\n"
        "data = pd.DataFrame({'value': [1, 2, 3]})\n"
        "assert data['value'].sum() == 6\n"
        "print('pandas: OK')",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "markdown",
        "## SciPy\n\nVerify that SciPy can perform numerical integration.",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "code",
        "from scipy import integrate\n\n"
        "result, _ = integrate.quad(lambda value: value ** 2, 0, 1)\n"
        "assert abs(result - 1 / 3) < 1e-10\n"
        "print('SciPy: OK')",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "markdown",
        "## ffmpeg\n\nVerify that ffmpeg can process generated media from a notebook.",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "code",
        "import subprocess\n\n"
        "subprocess.run(\n"
        "    ['ffmpeg', '-loglevel', 'error', '-f', 'lavfi', '-i',\n"
        "     'color=c=black:s=16x16:d=0.1', '-f', 'null', '-'],\n"
        "    check=True,\n"
        ")\n"
        "print('ffmpeg: OK')",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "markdown",
        "## ImageMagick\n\nVerify that ImageMagick can create an image from a notebook.",
        timeout=transition_timeout,
    )
    await add_cell(
        page,
        "code",
        "import subprocess\n"
        "import tempfile\n"
        "from pathlib import Path\n\n"
        "directory = tempfile.TemporaryDirectory()\n"
        "image = Path(directory.name) / 'image.png'\n"
        "subprocess.run(\n"
        "    ['convert', '-size', '16x16', 'xc:white', str(image)],\n"
        "    check=True,\n"
        ")\n"
        "assert image.stat().st_size > 0\n"
        "directory.cleanup()\n"
        "print('ImageMagick: OK')",
        timeout=transition_timeout,
    )
    print("✓ Runtime dependency cells created")

await run_pw(_step_create_test_cells)

## Run and verify runtime dependency checks

In [ ]:
async def _step_run_test_cells(page):
    await run(page, timeout=transition_timeout)

    expected_outputs = {
        2: "pandas: OK",
        4: "SciPy: OK",
        6: "ffmpeg: OK",
        8: "ImageMagick: OK",
    }
    for cell_index, expected_output in expected_outputs.items():
        cell = await get_cell(page, cell_index, timeout=transition_timeout)
        output = cell.locator('.jp-OutputArea-output')
        output_text = await output.text_content(timeout=transition_timeout)
        assert output_text is not None
        assert output_text.strip() == expected_output, (
            f"Unexpected output for cell {cell_index}: {output_text}"
        )

    print("✓ Runtime dependencies verified")

await run_pw(_step_run_test_cells)

## Save the notebook

In [ ]:
async def _step_save_notebook(page):
    await save_notebook(page)
    print("✓ Notebook saved")

await run_pw(_step_save_notebook)

## Cleanup

In [ ]:
await finish_pw_context()

In [ ]:
!rm -rf {work_dir}